# 🌱 Beginner Project 1: Data Cleaning & Text Filtering

Welcome to **Beginner Project 1**! In this project, you will learn the essentials of text data cleaning and filtering in Python using `pandas`, `datasets` (Hugging Face), and `valx` (a Python package for text moderation and profanity detection).

### 🎯 Objectives
- Load a dataset from Hugging Face (`dair-ai/emotion`).
- Convert dataset splits into Pandas DataFrames.
- Perform basic data quality checks (missing values, duplicates).
- Clean text data by stripping whitespace and normalizing text.
- Use `valx` to detect and filter profanity or sensitive content.


In [ ]:
# Step 1: Install required dependencies
!pip install -q datasets pandas valx


In [ ]:
# Step 2: Import libraries
import pandas as pd
from datasets import load_dataset
import valx

print("ValX version/module loaded successfully!")


In [ ]:
# Step 3: Load dataset from Hugging Face
# We load the 'dair-ai/emotion' dataset which contains text and emotion label annotations
dataset = load_dataset("dair-ai/emotion", split="train[:200]")

# Convert the Hugging Face dataset split into a Pandas DataFrame
df = pd.DataFrame(dataset)

# Display the first 5 rows
print(f"Dataset shape: {df.shape}")
df.head()


In [ ]:
# Step 4: Initial Data Quality Checks
# Check for null values
print("--- Missing Values ---")
print(df.isnull().sum())

# Check for duplicates
print("\n--- Duplicate Rows ---")
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# View basic statistics on text length
df['text_length'] = df['text'].apply(len)
print("\n--- Text Length Summary ---")
print(df['text_length'].describe())


In [ ]:
# Step 5: Text Cleaning Functions
def clean_text(text):
    """
    Basic text cleaning function:
    - Strips leading and trailing whitespaces
    - Converts text to lowercase
    """
    if not isinstance(text, str):
        return ""
    text = text.strip()
    return text

# Apply cleaning
df['cleaned_text'] = df['text'].apply(clean_text)
df[['text', 'cleaned_text']].head()


In [ ]:
# Step 6: Text Filtering using ValX
# Demonstration of ValX profanity detection and removal functions
sample_texts = [
    "This is a wonderful and clean dataset for beginners!",
    "This text contains a damn word that should be filtered.",
    "Data science with Python is awesome!"
]

print("--- ValX Demonstration ---")
for sample in sample_texts:
    is_profane = valx.detect_profanity(sample)
    cleaned_sample = valx.remove_profanity(sample)
    print(f"Original: '{sample}'")
    print(f"Contains Profanity? {is_profane}")
    print(f"Cleaned: '{cleaned_sample}'\n")

# Apply profanity flag check to our DataFrame
df['has_profanity'] = df['cleaned_text'].apply(lambda x: bool(valx.detect_profanity(x)))
profane_count = df['has_profanity'].sum()
print(f"Total rows flagged for profanity in dataset: {profane_count}")


In [ ]:
# Step 7: Filter dataset and export cleaned results
# Keep only clean texts without profanity
clean_df = df[df['has_profanity'] == False].copy()

print(f"Original Dataset Count: {len(df)}")
print(f"Cleaned Dataset Count:  {len(clean_df)}")

# Preview final cleaned dataset
clean_df[['text', 'label', 'text_length']].head()
